In [0]:

from pyspark.sql import functions as F
volume_path = "/Volumes/document_ai_catalog/document_processing/document_files"
input_path = f"{volume_path}/input/"
binary_df = (
    spark.read
    .format("binaryFile")
    .load(input_path)
)
display(binary_df.select("path", "length"))


path,length
dbfs:/Volumes/document_ai_catalog/document_processing/document_files/input/student_academic_report.docx,37419


In [0]:
binary_df.createOrReplaceTempView("binary_documents")
parsed_df = spark.sql(f"""
                      select path,
                      ai_parse_document(
                          content,
                          MAP('version','2.0')
                      ) as parsed
                      from binary_documents""")
display(parsed_df)

path parsed dbfs:/Volumes/document_ai_catalog/document_processing/document_files/input/student_academic_report.docx {"document":{"elements":[{"bbox":[{"coord":[265,152,1435,222],"page_id":0}],"confidence":1,"content":"Student Academic Performance Report","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[248,286,1370,383],"page_id":0}],"confidence":1,"content":"Programme: B.E. Computer Science and Engineering | Semester: 5\nThis sample document is designed for testing automated table extraction and classification.","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[248,458,626,495],"page_id":0}],"confidence":1,"content":"Table 1: Student Marks","description":null,"id":2,"type":"section_header"},{"bbox":[{"coord":[248,506,1452,772],"page_id":0}],"confidence":0.9994,"content":" Register Number Student Name Database Management (100) Computer Networks (100) Operating Systems (100) Total (300) 23CS101 Ananya 88 91 84 263 23CS102 Bharath 76 82 79 237 23CS103 Charan 92 89 95 276 23CS104 Divya 81 87 90 258 23CS105 Ezhil 69 74 72 215 ","description":null,"id":3,"type":"table"},{"bbox":[{"coord":[248,880,711,917],"page_id":0}],"confidence":1,"content":"Table 2: Student Attendance","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[248,928,1452,1164],"page_id":0}],"confidence":0.9995,"content":" Register Number Student Name Working Days Days Present Attendance Percentage 23CS101 Ananya 90 86 95.56 23CS102 Bharath 90 80 88.89 23CS103 Charan 90 89 98.89 23CS104 Divya 90 84 93.33 23CS105 Ezhil 90 77 85.56 ","description":null,"id":5,"type":"table"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"eb96164a-d1c9-4422-9d4d-398005dd850e","version":"2.0"}}

In [0]:
display(parsed_df.select("parsed"))

parsed {"document":{"elements":[{"bbox":[{"coord":[265,152,1435,224],"page_id":0}],"confidence":1,"content":"Student Academic Performance Report","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[246,284,1370,385],"page_id":0}],"confidence":1,"content":"Programme: B.E. Computer Science and Engineering | Semester: 5\nThis sample document is designed for testing automated table extraction and classification.","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[246,455,627,495],"page_id":0}],"confidence":1,"content":"Table 1: Student Marks","description":null,"id":2,"type":"section_header"},{"bbox":[{"coord":[246,504,1452,772],"page_id":0}],"confidence":0.9995,"content":" Register Number Student Name Database Management (100) Computer Networks (100) Operating Systems (100) Total (300) 23CS101 Ananya 88 91 84 263 23CS102 Bharath 76 82 79 237 23CS103 Charan 92 89 95 276 23CS104 Divya 81 87 90 258 23CS105 Ezhil 69 74 72 215 ","description":null,"id":3,"type":"table"},{"bbox":[{"coord":[246,878,712,917],"page_id":0}],"confidence":1,"content":"Table 2: Student Attendance","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[246,926,1452,1164],"page_id":0}],"confidence":0.9996,"content":" Register Number Student Name Working Days Days Present Attendance Percentage 23CS101 Ananya 90 86 95.56 23CS102 Bharath 90 80 88.89 23CS103 Charan 90 89 98.89 23CS104 Divya 90 84 93.33 23CS105 Ezhil 90 77 85.56 ","description":null,"id":5,"type":"table"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"e1309d88-d93b-46e3-95fa-cc6fc3cc79ff","version":"2.0"}}

In [0]:
parsed_df.createOrReplaceTempView("parsed_documents")


In [0]:
%sql
select 
element.pos AS table_position,
element.value:content::STRING as table_html,
element.value:confidence::DOUBLE as confidence
from parsed_documents,
lateral variant_explode(parsed:document:elements) as element
where element.value:type::STRING='table'
order by table_position;

table_position table_html confidence 3 Register Number Student Name Database Management (100) Computer Networks (100) Operating Systems (100) Total (300) 23CS101 Ananya 88 91 84 263 23CS102 Bharath 76 82 79 237 23CS103 Charan 92 89 95 276 23CS104 Divya 81 87 90 258 23CS105 Ezhil 69 74 72 215 0.9995 5 Register Number Student Name Working Days Days Present Attendance Percentage 23CS101 Ananya 90 86 95.56 23CS102 Bharath 90 80 88.89 23CS103 Charan 90 89 98.89 23CS104 Divya 90 84 93.33 23CS105 Ezhil 90 77 85.56 0.9996

In [0]:
tables_df = spark.sql("""
    SELECT
        element.pos AS table_position,
        element.value:content::STRING AS table_html
    FROM parsed_documents,
    LATERAL variant_explode(parsed:document:elements) AS element
    WHERE element.value:type::STRING = 'table'
    ORDER BY table_position
""")

display(tables_df)

table_position table_html 3 Register Number Student Name Database Management (100) Computer Networks (100) Operating Systems (100) Total (300) 23CS101 Ananya 88 91 84 263 23CS102 Bharath 76 82 79 237 23CS103 Charan 92 89 95 276 23CS104 Divya 81 87 90 258 23CS105 Ezhil 69 74 72 215 5 Register Number Student Name Working Days Days Present Attendance Percentage 23CS101 Ananya 90 86 95.56 23CS102 Bharath 90 80 88.89 23CS103 Charan 90 89 98.89 23CS104 Divya 90 84 93.33 23CS105 Ezhil 90 77 85.56

In [0]:
display(spark.sql("SELECT current_catalog(), current_schema()"))

current_catalog(),current_schema()
workspace,default


In [0]:
test_df = spark.sql("""
    SELECT
        ai_query(
            'system.ai.gpt-oss-120b',
            'Return a JSON object with two fields:
             status and message.
             Set status to success and message to
             Model connection is working.'
        ) AS response
""")

display(test_df)

response
"```json { ""status"": ""success"", ""message"": ""Model connection is working."" } ```"


In [0]:
tables_df.createOrReplaceTempView("extracted_tables")
schema_df = spark.sql("""
    SELECT
        table_position,
        ai_query(
            'system.ai.gpt-oss-120b',
            CONCAT(
                'Analyze this HTML table and generate a JSON Schema ',
                'for extracting all its data rows. ',
                'Use the table headers as field names. ',
                'Infer suitable data types from the headers and sample values. ',
                'Keep identifiers such as register numbers as strings, ',
                'even if they contain only digits. ',
                'Do not invent columns. ',
                'Use only string, integer, number, boolean or string ',
                'for dates. ',
                'Return only valid JSON, without Markdown or explanations. ',
                'Use this structure: ',
                '{"type":"object","properties":{"rows":{"type":"array",',
                '"items":{"type":"object","properties":{...}}}}}. ',
                'HTML TABLE: ',
                table_html
            )
        ) AS generated_schema
    FROM extracted_tables
""")

display(schema_df)

table_position,generated_schema
3,"{""type"":""object"",""properties"":{""rows"":{""type"":""array"",""items"":{""type"":""object"",""properties"":{""Register Number"":{""type"":""string""},""Student Name"":{""type"":""string""},""Database Management (100)"":{""type"":""integer""},""Computer Networks (100)"":{""type"":""integer""},""Operating Systems (100)"":{""type"":""integer""},""Total (300)"":{""type"":""integer""}}}}}}"
5,"{""type"":""object"",""properties"":{""rows"":{""type"":""array"",""items"":{""type"":""object"",""properties"":{""Register Number"":{""type"":""string""},""Student Name"":{""type"":""string""},""Working Days"":{""type"":""integer""},""Days Present"":{""type"":""integer""},""Attendance Percentage"":{""type"":""number""}},""required"":[""Register Number"",""Student Name"",""Working Days"",""Days Present"",""Attendance Percentage""]}}}}"


In [0]:
import json
def clean_schema(schema):
    if isinstance(schema, dict):
        schema.pop("required", None)
        for value in schema.values():
            clean_schema(value)
    elif isinstance(schema, list):
        for item in schema:
            clean_schema(item)
    return schema
schema = schema_df.filter(
    "table_position = 5"
).first()["generated_schema"]
schema = json.loads(schema)
schema = clean_schema(schema)
generated_schema = json.dumps(schema["properties"])
print(generated_schema)

{"rows": {"type": "array", "items": {"type": "object", "properties": {"Register Number": {"type": "string"}, "Student Name": {"type": "string"}, "Working Days": {"type": "integer"}, "Days Present": {"type": "integer"}, "Attendance Percentage": {"type": "number"}}}}}


In [0]:
import json
import re
schema = json.loads(generated_schema)
column_mapping = {}
def fix_property_names(node):
    if isinstance(node, dict):

        if "properties" in node:
            updated_properties = {}

            for old_name, value in node["properties"].items():

                new_name = re.sub(
                    r"[^\w.-]",
                    "_",
                    old_name,
                    flags=re.UNICODE
                ).lstrip("_")

                column_mapping[new_name] = old_name

                updated_properties[new_name] = fix_property_names(value)

            node["properties"] = updated_properties

        for key in node:
            if key != "properties":
                node[key] = fix_property_names(node[key])

    return node


schema = fix_property_names(schema)
generated_schema = json.dumps(schema)
print(generated_schema)
print(column_mapping)

{"rows": {"type": "array", "items": {"type": "object", "properties": {"Register_Number": {"type": "string"}, "Student_Name": {"type": "string"}, "Working_Days": {"type": "integer"}, "Days_Present": {"type": "integer"}, "Attendance_Percentage": {"type": "number"}}}}}
{'Register_Number': 'Register Number', 'Student_Name': 'Student Name', 'Working_Days': 'Working Days', 'Days_Present': 'Days Present', 'Attendance_Percentage': 'Attendance Percentage'}


In [0]:
table = tables_df.filter("table_position = 5").first()
table_html = table["table_html"]
table_html_sql = table_html.replace("'", "''")
schema_sql = generated_schema.replace("'", "''")
result_df = spark.sql(f"""
    SELECT ai_extract(
        '{table_html_sql}',
        '{schema_sql}',
        MAP('version', '2.1')
    ) AS extracted
""")

display(result_df)

extracted
"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""rows"":[{""Attendance_Percentage"":{""value"":95.56},""Days_Present"":{""value"":86},""Register_Number"":{""value"":""23CS101""},""Student_Name"":{""value"":""Ananya""},""Working_Days"":{""value"":90}},{""Attendance_Percentage"":{""value"":88.89},""Days_Present"":{""value"":80},""Register_Number"":{""value"":""23CS102""},""Student_Name"":{""value"":""Bharath""},""Working_Days"":{""value"":90}},{""Attendance_Percentage"":{""value"":98.89},""Days_Present"":{""value"":89},""Register_Number"":{""value"":""23CS103""},""Student_Name"":{""value"":""Charan""},""Working_Days"":{""value"":90}},{""Attendance_Percentage"":{""value"":93.33},""Days_Present"":{""value"":84},""Register_Number"":{""value"":""23CS104""},""Student_Name"":{""value"":""Divya""},""Working_Days"":{""value"":90}},{""Attendance_Percentage"":{""value"":85.56},""Days_Present"":{""value"":77},""Register_Number"":{""value"":""23CS105""},""Student_Name"":{""value"":""Ezhil""},""Working_Days"":{""value"":90}}]}}"


In [0]:
extracted_data = result_df.first()["extracted"].toPython()
rows = extracted_data["response"]["rows"]
clean_rows = []
for row in rows:
    clean_row = {}
    for column, value in row.items():
        clean_row[column] = value["value"]
    clean_rows.append(clean_row)
final_df = spark.createDataFrame(clean_rows)
display(final_df)

Attendance_Percentage,Days_Present,Register_Number,Student_Name,Working_Days
95.560000000000000000,86,23CS101,Ananya,90
88.890000000000000000,80,23CS102,Bharath,90
98.890000000000000000,89,23CS103,Charan,90
93.330000000000000000,84,23CS104,Divya,90
85.560000000000000000,77,23CS105,Ezhil,90


In [0]:
import json
import re

def clean_schema_names(node):

    if isinstance(node, dict):

        node.pop("required", None)

        if "properties" in node:

            updated_properties = {}

            for name, value in node["properties"].items():

                new_name = re.sub(r"[^\w.-]", "_", name)
                new_name = new_name.lstrip("_")

                updated_properties[new_name] = clean_schema_names(value)

            node["properties"] = updated_properties

        for key in node:
            if key != "properties":
                node[key] = clean_schema_names(node[key])

    elif isinstance(node, list):

        for i in range(len(node)):
            node[i] = clean_schema_names(node[i])

    return node


def extract_table(table_position):

    table = tables_df.filter(
        f"table_position = {table_position}"
    ).first()

    schema_row = schema_df.filter(
        f"table_position = {table_position}"
    ).first()

    table_html = table["table_html"]

    schema = json.loads(schema_row["generated_schema"])

    schema = clean_schema_names(schema)

    generated_schema = json.dumps(schema["properties"])

    table_html = table_html.replace("'", "''")
    generated_schema = generated_schema.replace("'", "''")

    result_df = spark.sql(f"""
        SELECT ai_extract(
            '{table_html}',
            '{generated_schema}',
            MAP('version', '2.1')
        ) AS extracted
    """)

    extracted_data = result_df.first()["extracted"].toPython()
    if extracted_data["error_message"] is not None:
        raise Exception(extracted_data["error_message"])
    rows = extracted_data["response"]["rows"]
    clean_rows = []
    for row in rows:
        clean_row = {}
        for column, value in row.items():
            clean_row[column] = value["value"]

        clean_rows.append(clean_row)

    return spark.createDataFrame(clean_rows)

In [0]:
all_tables = {}
for table in tables_df.orderBy("table_position").collect():

    position = table["table_position"]

    final_df = extract_table(position)

    all_tables[position] = final_df

    print(f"Extracted Table at position {position}")

    display(final_df)

Extracted Table at position 3


Computer_Networks__100_,Database_Management__100_,Operating_Systems__100_,Register_Number,Student_Name,Total__300_
91,88,84,23CS101,Ananya,263
82,76,79,23CS102,Bharath,237
89,92,95,23CS103,Charan,276
87,81,90,23CS104,Divya,258
74,69,72,23CS105,Ezhil,215


Extracted Table at position 5


Attendance_Percentage,Days_Present,Register_Number,Student_Name,Working_Days
95.560000000000000000,86,23CS101,Ananya,90
88.890000000000000000,80,23CS102,Bharath,90
98.890000000000000000,89,23CS103,Charan,90
93.330000000000000000,84,23CS104,Divya,90
85.560000000000000000,77,23CS105,Ezhil,90


In [0]:
output_path = volume_path + "/output/"
document_name = "student_academic_report"
for position, df in all_tables.items():
    file_name = f"{document_name}_table_{position}.csv"
    temp_path = output_path + "temp_" + str(position)
    final_path = output_path + file_name
    df.coalesce(1).write \
        .option("header", True) \
        .mode("overwrite") \
        .csv(temp_path)
    csv_file = [
        file.path for file in dbutils.fs.ls(temp_path)
        if file.name.endswith(".csv")
    ][0]
    dbutils.fs.mv(csv_file, final_path)
    dbutils.fs.rm(temp_path, True)
    print(f"Saved: {file_name}")

Saved: student_academic_report_table_3.csv
Saved: student_academic_report_table_5.csv


In [0]:
import json
labels = {
    "Academic_Report": "Student records, academic performance, marks, attendance, examinations and educational reports.",
    "Employee_Information": "Employee records, staff details, employee performance, HR information and workforce reports.",
    "Sales_and_Finance": "Sales reports, revenue, expenses, financial statements, transactions and business accounts.",
    "Technical_Documentation": "Technical specifications, software documentation, system reports and engineering documents.",
    "Other": "Documents that do not fit into any of the above categories."
}
labels_json = json.dumps(labels)

In [0]:
labels_sql = labels_json.replace("'", "''")

queries = []

for table in tables_df.orderBy("table_position").collect():

    position = table["table_position"]

    table_html = table["table_html"].replace("'", "''")

    queries.append(f"""
        SELECT
            {position} AS table_position,
            ai_classify(
                '{table_html}',
                '{labels_sql}',
                MAP(
                    'version', '2.1',
                    'enableConfidenceScores', 'true',
                    'enableRationales', 'true'
                )
            ) AS classification
    """)

classification_df = spark.sql(
    " UNION ALL ".join(queries)
)

display(classification_df)

table_position,classification
3,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""rationale"":""The table lists student register numbers, names, and marks for courses, which are academic performance records."",""value"":""Academic_Report""}]}"
5,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.92,""rationale"":""The table lists register numbers, student names, working days, days present, and attendance percentages, which are typical elements of a student attendance or academic performance report."",""value"":""Academic_Report""}]}"


In [0]:
classification_results = []

for row in classification_df.collect():

    data = row["classification"].toPython()

    if data["error_message"] is not None:
        raise Exception(data["error_message"])

    result = data["response"][0]

    position = row["table_position"]

    classification_results.append({
        "Table_Position": position,
        "Output_File": f"student_academic_report_table_{position}.csv",
        "Category": result["value"]
    })

classification_output_df = spark.createDataFrame(
    classification_results
)

display(classification_output_df)

Category,Output_File,Table_Position
Academic_Report,student_academic_report_table_3.csv,3
Academic_Report,student_academic_report_table_5.csv,5


In [0]:
tracking_path = volume_path + "/tracking/category_tracking.txt"
new_entries = []
for row in classification_results:
    entry = f"{row['Output_File']} : {row['Category']}"
    new_entries.append(entry)
new_content = "\n".join(new_entries)
if dbutils.fs.ls(volume_path + "/tracking/"):

    try:
        existing_content = dbutils.fs.head(tracking_path)

    except Exception:
        existing_content = ""

    if existing_content.strip():
        final_content = existing_content.rstrip() + "\n" + new_content
    else:
        final_content = new_content

else:
    final_content = new_content

dbutils.fs.put(
    tracking_path,
    final_content + "\n",
    overwrite=True
)

print("Tracking file updated successfully!")

Wrote 108 bytes.
Tracking file updated successfully!
